### Extract 1km Data
Extract data from various 1km resolution netCDF files.

### Load datasets
I am not sure what platform I will run this on, so I will specify paths explicitly.

In [2]:
# Paths

platform = 'alpine'

if platform == 'desktop':
    SNOTEL_PATH = "C:/Users/clmbn/NMT_PhD/data/snotel/bcqc_data_v2"
    DATA_PATH = "C:/Users/clmbn/NMT_PhD/data/ESM1k_land/"
elif platform == 'alpine':
    SNOTEL_PATH = "/projects/ccox1@xsede.org/data/snotel/bcqc_data_v2/"
    DATA_PATH = "/pl/active/snotelmodelling/global1km_data"
else:
    print('Error: wrong platform specified')

In [3]:
# Imports
import pandas as pd
import xarray as xr
import numpy as np

### Load SNOTEL sites

In [4]:
# Load SNOTEL sites CSV
snotel_df = pd.read_csv(f"{SNOTEL_PATH}/SNOTEL_summary.csv")
print(f"Loaded {len(snotel_df)} SNOTEL sites")
snotel_df.head()

Loaded 829 SNOTEL sites


,Num,SNOTEL ID,SNOTEL State,SNOTEL Name,SNOTEL Elev (ft),SNOTEL Lat (),SNOTEL Lon (),Data Starting Date,Data Ending Date,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,1,1000,OR,Annie Springs,6010,42.87,-122.17,9/15/2000,9/30/2018,NaN,NaN,NaN
1,2,1001,AK,Long Lake,850,58.19,-133.83,10/1/1999,9/30/2018,NaN,NaN,NaN
2,3,1003,AK,Mcneil Canyon,1320,59.74,-151.25,10/1/1986,9/30/2018,NaN,NaN,NaN
3,4,1005,CO,Ute Creek,10650,37.61,-105.37,8/15/2000,9/30/2018,NaN,NaN,NaN
4,5,1006,NV,Lewis Peak,7400,40.36,-116.86,8/23/2000,9/30/2018,NaN,NaN,NaN


### NetCDF manipulation functions

In [5]:
# Load netCDF metadata
def display_netcdf_metadata(file_path):
    """
    Display metadata for a netCDF file including dimensions, coordinates, variables, and attributes.
    
    Parameters:
    -----------
    file_path : str
        Path to the netCDF file
    
    Returns:
    --------
    xarray.Dataset
        The opened dataset
    """
    # Open the dataset
    ds = xr.open_dataset(file_path)
    
    print(f"=" * 80)
    print(f"NetCDF File: {file_path}")
    print(f"=" * 80)
    
    # Display dimensions
    print("\nDIMENSIONS:")
    print("-" * 40)
    for dim_name, dim_size in ds.dims.items():
        print(f"  {dim_name}: {dim_size}")
    
    # Display coordinates
    print("\nCOORDINATES:")
    print("-" * 40)
    for coord_name in ds.coords:
        coord = ds.coords[coord_name]
        print(f"  {coord_name}:")
        print(f"    - Shape: {coord.shape}")
        print(f"    - Dtype: {coord.dtype}")
        if len(coord.shape) == 1 and coord.shape[0] > 0:
            print(f"    - Range: [{coord.values.min():.4f}, {coord.values.max():.4f}]")
    
    # Display data variables
    print("\nDATA VARIABLES:")
    print("-" * 40)
    for var_name in ds.data_vars:
        var = ds[var_name]
        print(f"  {var_name}:")
        print(f"    - Dimensions: {var.dims}")
        print(f"    - Shape: {var.shape}")
        print(f"    - Dtype: {var.dtype}")
        # Print attributes if they exist
        if var.attrs:
            print(f"    - Attributes:")
            for attr_name, attr_value in var.attrs.items():
                print(f"        {attr_name}: {attr_value}")
    
    # Display global attributes
    if ds.attrs:
        print("\nGLOBAL ATTRIBUTES:")
        print("-" * 40)
        for attr_name, attr_value in ds.attrs.items():
            print(f"  {attr_name}: {attr_value}")
    
    print("\n" + "=" * 80)
    
    return ds

In [6]:
# Extract data from netCDF for site locations
def extract_netcdf_values(sites, variable_name, netcdf_file, lat_col='SNOTEL Lat ()', lon_col='SNOTEL Lon ()'):
    """
    Extract values from a netCDF file at specified site locations using vectorized operations.
    
    Parameters:
    -----------
    sites : pd.DataFrame
        Dataframe containing site information with latitude and longitude columns
    variable_name : str
        Name of the variable to extract from the netCDF file
    netcdf_file : str or xarray.Dataset
        Path to the netCDF file or an already opened xarray Dataset
    lat_col : str, optional
        Name of the latitude column in the sites dataframe (default: 'SNOTEL Lat ()')
    lon_col : str, optional
        Name of the longitude column in the sites dataframe (default: 'SNOTEL Lon ()')
    
    Returns:
    --------
    np.array
        Array of extracted values for each site
    """
    # Open dataset if a file path is provided
    if isinstance(netcdf_file, str):
        ds = xr.open_dataset(netcdf_file)
    else:
        ds = netcdf_file
    
    # Get arrays of lat/lon values
    lats = sites[lat_col].values
    lons = sites[lon_col].values
    
    # Use vectorized selection - pass all coordinates at once
    # Create xarray DataArrays for the coordinates
    lat_da = xr.DataArray(lats, dims='sites')
    lon_da = xr.DataArray(lons, dims='sites')
    
    # Select all points at once using vectorized indexing
    points = ds[variable_name].sel(lat=lat_da, lon=lon_da, method='nearest')
    
    # Extract values as numpy array
    values = points.values
    
    return values

### Extract Data

In [7]:
output_data = snotel_df.loc[snotel_df['SNOTEL State'] != 'AK',['SNOTEL Name','SNOTEL Lat ()','SNOTEL Lon ()']].copy()


In [ ]:
output_data.head()
print(len(output_data))

#### Slope

In [ ]:
slope_ds = display_netcdf_metadata(f"{DATA_PATH}/slope.nc")

In [1]:
# Extract elevation values using the function
output_df['slope'] = extract_netcdf_values(
    sites=output_data,
    variable_name='SLOPE',
    netcdf_file=slope_ds
)

NameError: name 'extract_netcdf_values' is not defined

#### Aspect

In [11]:
aspect_ds = display_netcdf_metadata(f"{DATA_PATH}/aspect.nc")

NetCDF File: /pl/active/snotelmodelling/global1km_data/aspect.nc

DIMENSIONS:
----------------------------------------
  lat: 21600
  lon: 43200

COORDINATES:
----------------------------------------
  lat:
    - Shape: (21600,)
    - Dtype: float64
    - Range: [-89.9958, 89.9958]
  lon:
    - Shape: (43200,)
    - Dtype: float64
    - Range: [-179.9958, 179.9958]

DATA VARIABLES:
----------------------------------------
  ASPECT:
    - Dimensions: ('lat', 'lon')
    - Shape: (21600, 43200)
    - Dtype: float32
    - Attributes:
        standard_name: ground_slope_direction
        long_name: ground slope direction
        units: degree
        description: terrain face direction, degrees clockwise from 0° north

GLOBAL ATTRIBUTES:
----------------------------------------
  title: Global 1km Land Surface Parameters for Kilometer Scale Earth System Modeling
  version: 1.0
  institution: Atmospheric Sciences and Global Change Division, Pacific Northwest National Laboratory
  contact: Li

/tmp/ipykernel_2232601/278603443.py:26: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for dim_name, dim_size in ds.dims.items():


In [ ]:
output_df['aspect'] = extract_netcdf_values(
    sites=output_data,
    variable_name='ASPECT',
    netcdf_file=aspect_ds
)

### Save output